# Notebook 2 — URL features and spacing normalization

This notebook tests whether URL/structure features and normalization improve SMS spam classification. It uses training-only vectorizer fitting and the same deterministic split as Notebook 1.

## 1. Setup, load, and split

In [1]:
!pip -q install pandas numpy scikit-learn scipy joblib matplotlib
import re, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [2]:
DATA_PATH = "/content/sms_canonical.csv"
if not Path(DATA_PATH).exists():
    from google.colab import files
    uploaded = files.upload()
    csvs = [n for n in uploaded if n.lower().endswith(".csv")]
    if not csvs: raise FileNotFoundError("Upload sms_canonical.csv with text,label columns.")
    DATA_PATH = "/content/" + csvs[0]
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape, "Columns:", df.columns.tolist())
display(df.head())

Shape: (67008, 2) Columns: ['text', 'label']


,text,label
0,There be an update for your delivery CC 017281...,spam
1,watch your favorite english movies of all genr...,spam
2,aur what is the status for fms,ham
3,hi shalini sundi thank you for dialling speci...,spam
4,m tryin to understand too...,ham


In [3]:
if not {"text","label"}.issubset(df.columns):
    raise ValueError("Expected columns: text,label")
df = df[["text","label"]].dropna().copy()
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.lower().str.strip()
df = df[(df.text != "") & df.label.isin(["ham","spam"])]
before = len(df)
df = df.drop_duplicates(subset=["text","label"]).reset_index(drop=True)
print("Rows:", len(df), "| duplicates removed:", before-len(df))
display(df.label.value_counts().to_frame("count"))
X_train, X_temp, y_train, y_temp = train_test_split(df.text, df.label, test_size=.30, random_state=42, stratify=df.label)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=.50, random_state=42, stratify=y_temp)
print("Train/validation/test:", len(X_train), len(X_val), len(X_test))

Rows: 66986 | duplicates removed: 22


,count
label,
ham,40828
spam,26158


Train/validation/test: 46890 10048 10048


In [4]:
def metrics(name, y_true, y_pred):
    return {"model":name, "accuracy":accuracy_score(y_true,y_pred),
            "spam_precision":precision_score(y_true,y_pred,pos_label="spam",zero_division=0),
            "spam_recall":recall_score(y_true,y_pred,pos_label="spam",zero_division=0),
            "spam_f1":f1_score(y_true,y_pred,pos_label="spam",zero_division=0),
            "macro_f1":f1_score(y_true,y_pred,average="macro",zero_division=0)}

## 2. Define URL features

In [5]:
URL_RE = re.compile(r"(https?://|www\.|bit\.ly|tinyurl|t\.co|goo\.gl)", re.I)
DOMAIN_RE = re.compile(r"\b(?:[a-z0-9-]+\.)+(?:com|net|org|info|biz|co|io|me|ly|xyz|top|site)\b", re.I)
FEATURE_NAMES = ["url_marker","domain_marker","url_count","http_count","www_marker","bitly_marker","tinyurl_marker","tco_marker","digit_count","uppercase_count","special_punctuation_count","claim_word","winner_word","free_word","urgency_word","call_word","stop_word","message_length"]
def url_features(text):
    s = str(text); low = s.lower()
    urls = re.findall(r"(?:https?://\S+|www\.\S+)", low)
    return [int(bool(URL_RE.search(s))), int(bool(DOMAIN_RE.search(s))), len(urls),
            len(re.findall(r"\bhttps?://",low)), int("www." in low), int("bit.ly" in low),
            int("tinyurl" in low), int("t.co" in low), sum(c.isdigit() for c in s),
            sum(c.isupper() for c in s), sum(c in "!$%&*?" for c in s),
            int("claim" in low), int(("winner" in low) or ("won" in low)), int("free" in low),
            int(("urgent" in low) or ("hurry" in low)), int("call" in low), int("stop" in low), len(s)]
Xtr_url = np.asarray([url_features(x) for x in X_train], dtype=float)
Xte_url = np.asarray([url_features(x) for x in X_test], dtype=float)
print("Feature count:", len(FEATURE_NAMES), "matrix:", Xtr_url.shape)

Feature count: 18 matrix: (46890, 18)


## 3. Train word + character + URL model

In [6]:
wv = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, sublinear_tf=True)
cv = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=2, sublinear_tf=True)
Wtr, Wte = wv.fit_transform(X_train), wv.transform(X_test)
Ctr, Cte = cv.fit_transform(X_train), cv.transform(X_test)
Xtr_text = hstack([Wtr,Ctr]).tocsr(); Xte_text = hstack([Wte,Cte]).tocsr()
Xtr_all = hstack([Xtr_text,csr_matrix(Xtr_url)]).tocsr()
Xte_all = hstack([Xte_text,csr_matrix(Xte_url)]).tocsr()
url_model = LinearSVC(class_weight="balanced", random_state=42).fit(Xtr_all,y_train)
url_result = metrics("word_char_url_features", y_test, url_model.predict(Xte_all))
print(url_result)

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


{'model': 'word_char_url_features', 'accuracy': 0.9847730891719745, 'spam_precision': 0.9750566893424036, 'spam_recall': 0.9862385321100917, 'spam_f1': 0.9806157354618016, 'macro_f1': 0.9840389174727675}


## 4. URL prevalence by class

In [7]:
prevalence = pd.DataFrame({"label":df.label, "has_url":df.text.apply(lambda t: bool(URL_RE.search(t)))})
display((prevalence.groupby("label").has_url.mean()*100).rename("messages_with_url_percent").reset_index())

,label,messages_with_url_percent
0,ham,0.061232
1,spam,24.103525


## 5. Train normalized word + character model

In [8]:
def normalize_obfuscated_text(text):
    s = str(text).lower()
    pattern = re.compile(r"(?<!\S)(?:[a-z0-9]\s+){3,}[a-z0-9](?!\S)", re.I)
    return pattern.sub(lambda m: re.sub(r"\s+", "", m.group(0)), s)
Xtr_norm = X_train.map(normalize_obfuscated_text)
Xte_norm = X_test.map(normalize_obfuscated_text)
wv_n = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, sublinear_tf=True)
cv_n = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=2, sublinear_tf=True)
Wtr_n, Wte_n = wv_n.fit_transform(Xtr_norm), wv_n.transform(Xte_norm)
Ctr_n, Cte_n = cv_n.fit_transform(Xtr_norm), cv_n.transform(Xte_norm)
Xtr_n = hstack([Wtr_n,Ctr_n]).tocsr(); Xte_n = hstack([Wte_n,Cte_n]).tocsr()
norm_model = LinearSVC(class_weight="balanced", random_state=42).fit(Xtr_n,y_train)
norm_result = metrics("normalized_word_char", y_test, norm_model.predict(Xte_n))
print(norm_result)

{'model': 'normalized_word_char', 'accuracy': 0.9932324840764332, 'spam_precision': 0.9948665297741273, 'spam_recall': 0.9877675840978594, 'spam_f1': 0.991304347826087, 'macro_f1': 0.9928825421111536}


## 6. Compare and save

In [9]:
results = pd.DataFrame([url_result,norm_result]); display(results)
OUT = Path("/content/sms_spam_outputs"); OUT.mkdir(exist_ok=True)
joblib.dump({"word_vectorizer":wv,"char_vectorizer":cv,"classifier":url_model,"feature_names":FEATURE_NAMES},OUT/"word_char_url_model.joblib")
joblib.dump({"word_vectorizer":wv_n,"char_vectorizer":cv_n,"classifier":norm_model,"normalization":"normalize_obfuscated_text"},OUT/"word_char_normalized_model.joblib")
results.to_csv(OUT/"url_normalization_metrics.csv",index=False)
print("Saved to",OUT)

,model,accuracy,spam_precision,spam_recall,spam_f1,macro_f1
0,word_char_url_features,0.984773,0.975057,0.986239,0.980616,0.984039
1,normalized_word_char,0.993232,0.994867,0.987768,0.991304,0.992883


Saved to /content/sms_spam_outputs


## Research note

URL features can help with some messages, but URL absence does not mean a message is safe. Compare clean-test metrics with adversarial robustness metrics separately.